# Auto-Caption Demo Notebook

This notebook demonstrates how to use the Auto-Caption library for emotion-aware caption generation in Jupyter.

## Features Covered:
- Installation and setup
- Basic caption generation
- Emotion detection from video
- Styled subtitle generation
- Video merging with captions
- Interactive visualization

## 1. Installation

First, let's install the auto-caption package. If you're running this from the project directory:

In [ ]:
# Install from local directory (if you're in the auto-caption project)
!pip install -e ../..

# Or install specific requirements
# !pip install -r ../../requirements.txt

## 2. Import Required Modules

In [ ]:
# Import auto-caption modules
from auto_caption import (
    CaptionGenerator,
    EmotionDetector,
    CaptionStyler,
    VideoMerger,
    EmotionCategory,
    StyleIntensity,
    Platform,
    WordTimingMode
)
from auto_caption.subtitle import ASSGenerator
from auto_caption.visualization import EmotionVisualizer

# Import other useful libraries
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display, HTML
import pandas as pd

## 3. Basic Caption Generation

Let's start with basic speech-to-text caption generation:

In [ ]:
# Initialize caption generator
generator = CaptionGenerator(
    model_name="base",  # Can be: tiny, base, small, medium, large
    language="en",
    device="cuda"  # or "cpu"
)

# Generate captions from video
video_path = "sample_video.mp4"  # Replace with your video path

# Check if video exists
if os.path.exists(video_path):
    result = generator.generate_captions(
        video_path=video_path,
        word_level=True,
        verbose=True
    )
    
    # Display results
    print(f"Video duration: {result['duration']:.2f} seconds")
    print(f"Number of segments: {len(result['segments'])}")
    print("\nFirst few segments:")
    for i, seg in enumerate(result['segments'][:3]):
        print(f"{i+1}. [{seg['start']:.2f}s - {seg['end']:.2f}s]: {seg['text']}")
else:
    print(f"Video file not found: {video_path}")
    print("Please provide a valid video path")

## 4. Emotion Detection

Now let's detect emotions from the video:

In [ ]:
# Initialize emotion detector
emotion_detector = EmotionDetector(
    model_name="emotion_model.pth",  # Will download if not present
    device="cuda",  # or "cpu"
    confidence_threshold=0.7
)

# Process video for emotions
if os.path.exists(video_path):
    # Detect emotions with face tracking
    emotion_results = emotion_detector.process_video(
        video_path=video_path,
        sample_rate=0.5,  # Sample every 0.5 seconds
        show_progress=True
    )
    
    # Display emotion timeline
    print(f"\nDetected {len(emotion_results)} emotion samples")
    
    # Create emotion distribution
    emotion_counts = {}
    for result in emotion_results:
        emotion = result['emotion']
        emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1
    
    # Plot emotion distribution
    plt.figure(figsize=(10, 6))
    emotions = list(emotion_counts.keys())
    counts = list(emotion_counts.values())
    plt.bar(emotions, counts)
    plt.xlabel('Emotion')
    plt.ylabel('Frame Count')
    plt.title('Emotion Distribution in Video')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Caption Styling Based on Emotions

Let's style captions based on detected emotions:

In [ ]:
# Initialize caption styler
styler = CaptionStyler(
    default_intensity=StyleIntensity.MEDIUM,
    default_platform=Platform.GENERAL
)

# Example: Style some text with different emotions
sample_texts = [
    "This is amazing! I can't believe it!",
    "I'm feeling really down today.",
    "Oh yeah, that's totally going to work..."
]

emotions = [
    EmotionCategory.EXCITED,
    EmotionCategory.SAD,
    EmotionCategory.SARCASTIC
]

print("Emotion-based text styling examples:\n")
for text, emotion in zip(sample_texts, emotions):
    result = styler.style_caption(
        text=text,
        emotion=emotion,
        confidence=0.9
    )
    print(f"Original: {text}")
    print(f"Emotion: {emotion.value}")
    print(f"Styled: {result['formatted_text']}")
    print(f"Format: {result['format_type']}\n")

## 6. Generate Styled Subtitles

Create ASS (Advanced SubStation Alpha) subtitles with emotion-based styling:

In [ ]:
# Create sample caption data with emotions
caption_data = {
    "video_file": video_path,
    "duration": 30.0,
    "segments": [
        {
            "start": 0.0,
            "end": 3.0,
            "text": "Welcome to this amazing demo!",
            "emotion": "happy",
            "confidence": 0.9,
            "formatted_text": "Welcome to this AMAZING demo!"
        },
        {
            "start": 3.0,
            "end": 6.0,
            "text": "Let's explore emotion-aware captions",
            "emotion": "excited",
            "confidence": 0.85,
            "formatted_text": "Let's explore EMOTION-AWARE captions!!!"
        },
        {
            "start": 6.0,
            "end": 9.0,
            "text": "It adapts to your emotions",
            "emotion": "contemplative",
            "confidence": 0.8,
            "formatted_text": "It adapts to your emotions..."
        }
    ]
}

# Generate ASS subtitle file
ass_generator = ASSGenerator(
    platform=Platform.GENERAL,
    style_intensity=StyleIntensity.MEDIUM,
    video_resolution=(1920, 1080)
)

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

ass_file = ass_generator.generate_ass_file(
    caption_data=caption_data,
    output_path=os.path.join(output_dir, "styled_subtitles.ass"),
    title="Emotion-Aware Subtitles Demo"
)

print(f"Generated subtitle file: {ass_file}")

# Preview the ASS file content
with open(ass_file, 'r', encoding='utf-8') as f:
    content = f.read()
    print("\nFirst 500 characters of ASS file:")
    print(content[:500] + "...")

## 7. Merge Video with Styled Captions

Combine the video with emotion-styled captions:

In [ ]:
# Initialize video merger
merger = VideoMerger(
    platform=Platform.GENERAL,
    quality="high",
    verbose=True
)

# Merge video with captions
if os.path.exists(video_path) and os.path.exists(ass_file):
    output_video = os.path.join(output_dir, "video_with_captions.mp4")
    
    # Perform merge
    result = merger.merge_with_captions(
        video_path=video_path,
        caption_data=caption_data,
        output_path=output_video,
        subtitle_format="ass",
        style_intensity=StyleIntensity.MEDIUM
    )
    
    print(f"\nMerge complete!")
    print(f"Output video: {result['output_path']}")
    print(f"Processing time: {result['processing_time']:.2f} seconds")
    
    # Display the video in notebook (if file size is reasonable)
    if os.path.getsize(output_video) < 50 * 1024 * 1024:  # Less than 50MB
        display(Video(output_video, width=640, height=360))
else:
    print("Video or subtitle file not found")

## 8. Emotion Timeline Visualization

Visualize the emotion timeline of your video:

In [ ]:
# Create emotion timeline visualization
if 'emotion_results' in locals() and emotion_results:
    # Extract timeline data
    timestamps = [r['timestamp'] for r in emotion_results]
    emotions = [r['emotion'] for r in emotion_results]
    confidences = [r['confidence'] for r in emotion_results]
    
    # Create color map for emotions
    emotion_colors = {
        EmotionCategory.HAPPY.value: '#FFD700',
        EmotionCategory.SAD.value: '#4169E1',
        EmotionCategory.ANGRY.value: '#FF0000',
        EmotionCategory.EXCITED.value: '#FF1493',
        EmotionCategory.FEARFUL.value: '#9370DB',
        EmotionCategory.NEUTRAL.value: '#808080',
        EmotionCategory.SARCASTIC.value: '#32CD32',
        EmotionCategory.ANXIOUS.value: '#90EE90',
        EmotionCategory.CONTEMPLATIVE.value: '#DDA0DD'
    }
    
    # Create timeline plot
    plt.figure(figsize=(15, 8))
    
    # Plot emotion timeline
    for i, (ts, emotion, conf) in enumerate(zip(timestamps, emotions, confidences)):
        color = emotion_colors.get(emotion, '#000000')
        plt.scatter(ts, emotion, color=color, s=conf*200, alpha=0.7)
    
    plt.xlabel('Time (seconds)')
    plt.ylabel('Emotion')
    plt.title('Emotion Timeline')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Create emotion transition matrix
    print("\nEmotion Transitions:")
    transitions = {}
    for i in range(1, len(emotions)):
        prev_emotion = emotions[i-1]
        curr_emotion = emotions[i]
        key = f"{prev_emotion} → {curr_emotion}"
        transitions[key] = transitions.get(key, 0) + 1
    
    # Display top transitions
    sorted_transitions = sorted(transitions.items(), key=lambda x: x[1], reverse=True)
    for trans, count in sorted_transitions[:5]:
        print(f"{trans}: {count} times")

## 9. Export Results

Save the processed caption data for later use:

In [ ]:
# Export caption data to JSON
output_json = os.path.join(output_dir, "caption_data.json")

# Prepare export data
export_data = {
    "video_file": video_path,
    "duration": caption_data.get('duration', 0),
    "segments": caption_data.get('segments', []),
    "emotion_timeline": [
        {
            "timestamp": r['timestamp'],
            "emotion": r['emotion'],
            "confidence": r['confidence']
        }
        for r in emotion_results
    ] if 'emotion_results' in locals() else [],
    "metadata": {
        "platform": Platform.GENERAL.value,
        "style_intensity": StyleIntensity.MEDIUM.value,
        "language": "en",
        "processed_date": str(pd.Timestamp.now())
    }
}

# Save to JSON
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"Caption data exported to: {output_json}")

# Create summary DataFrame
if export_data['segments']:
    df = pd.DataFrame(export_data['segments'])
    print("\nCaption Summary:")
    print(f"Total segments: {len(df)}")
    print(f"Total duration: {df['end'].max():.2f} seconds")
    print(f"\nEmotion distribution:")
    print(df['emotion'].value_counts())

## 10. Interactive Demo

Try the caption styler with your own text:

In [ ]:
# Interactive styling demo
def style_text_interactive(text, emotion_name, intensity="medium"):
    """Style text based on emotion and intensity."""
    try:
        # Convert string to enum
        emotion = EmotionCategory[emotion_name.upper()]
        intensity_enum = StyleIntensity[intensity.upper()]
        
        # Create styler with specified intensity
        styler = CaptionStyler(default_intensity=intensity_enum)
        
        # Style the text
        result = styler.style_caption(
            text=text,
            emotion=emotion,
            confidence=0.9
        )
        
        # Display results
        print(f"Original: {text}")
        print(f"Emotion: {emotion.value} (Intensity: {intensity})")
        print(f"Styled: {result['formatted_text']}")
        print(f"Format Type: {result['format_type']}")
        
        return result
    except KeyError:
        print(f"Invalid emotion or intensity. Available emotions:")
        print([e.name.lower() for e in EmotionCategory])
        print(f"\nAvailable intensities: subtle, medium, intense")

# Try it out!
print("Try styling your own text with different emotions:\n")
print("Available emotions:", [e.name.lower() for e in EmotionCategory][:10], "...")
print("\nExample usage:")

# Example 1
style_text_interactive(
    "I just won the lottery!",
    "excited",
    "intense"
)

print("\n" + "-"*50 + "\n")

# Example 2
style_text_interactive(
    "Oh great, another meeting...",
    "sarcastic",
    "medium"
)

## Conclusion

This notebook demonstrated the key features of the Auto-Caption library:

1. **Speech Recognition**: Extract text from video audio
2. **Emotion Detection**: Analyze facial expressions for emotional context
3. **Caption Styling**: Format text based on detected emotions
4. **Subtitle Generation**: Create professional ASS subtitles with emotion-aware styling
5. **Video Merging**: Combine original video with styled captions

### Next Steps:
- Try with your own videos
- Experiment with different emotion intensities
- Customize the styling for specific platforms (TikTok, Instagram, etc.)
- Train custom emotion detection models
- Add word-by-word animation effects

For more examples and documentation, visit the [Auto-Caption GitHub repository](https://github.com/cds-id/auto-caption).